# Python 1: values and types

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-01-values-and-types.ipynb)

Python is the language most data engineers write first. Before you build a pipeline you need to know what a value is, how Python stores it, and which operations are safe. This chapter covers the built-in scalar types (`int`, `float`, `Decimal`, `str`, `bool` and `None`), the conversions between them, and the traps that give wrong answers on real data: integer division, floating-point error, money, text encodings and missing values.

Every rule is stated before the code that shows it, and each code cell's `assert` lines are the expected results. The notebook needs no installs. If an `assert` fails, your model of that rule is wrong, so read the comment next to it.
**Runtime:** Python 3.12 or later.

## Values, names and types

A **value** is an object with a type. A **name** is a label that points at a value. Assignment attaches the label to the value. It never copies the value.

`type(x)` returns the exact type of `x`. `isinstance(x, T)` also accepts subclasses, so it is the check to use in code. `bool` is a subclass of `int`, so `isinstance(True, int)` is `True`.

### Where you'll use it

- **Schema checks:** before loading a file, check each field's type. A `"12.0"` string where an integer belongs is one of the most common reasons a load fails.
- **Logs and debugging:** `repr(x)` shows the type and the exact value, quotes included, so a log line tells you what the pipeline received, not what you expected.

In [ ]:
count = 42
price = 19.99
label = "orders"
is_active = True
nothing = None

assert type(count) is int
assert type(price) is float
assert type(label) is str
assert type(is_active) is bool
assert type(nothing) is type(None)

# Two names, one object: assignment shares the value.
first = label
assert first is label

# isinstance accepts several types, and subclasses.
assert isinstance(count, (int, float))
assert isinstance(True, int) and not isinstance(True, str)

# repr shows quotes, so "42" and 42 are visibly different in a log.
assert repr("42") == "'42'" and repr(42) == "42"

## Integers and floats

- `int` has no size limit. Python never overflows: `2 ** 100` is exact.
- `float` is a 64-bit binary floating-point number. It stores the nearest binary fraction, so many decimal values, such as `0.1`, are only approximated.
- `/` always returns a `float`, even for `6 / 3`. `//` is floor division: it rounds toward negative infinity, not toward zero. `%` is the remainder, and it takes the sign of the divisor.
- `round(x)` rounds exact halves to the nearest **even** integer ("banker's rounding"). `round(2.5)` is `2`, not `3`.
- `int(x)` truncates toward zero. `int(-2.7)` is `-2`, but `math.floor(-2.7)` is `-3`.

In [ ]:
import math

assert 7 / 2 == 3.5 and 6 / 3 == 2.0 and type(6 / 3) is float
assert 7 // 2 == 3 and 7 % 2 == 1

# Floor division and remainder on negatives: floor, not truncation.
assert -7 // 2 == -4
assert -7 % 2 == 1          # the remainder takes the sign of the divisor
assert 7 % -2 == -1

# Truncation toward zero versus floor.
assert int(-2.7) == -2
assert math.floor(-2.7) == -3

# Banker's rounding on exact halves.
assert round(2.5) == 2 and round(3.5) == 4
assert round(2.675, 2) == 2.67      # 2.675 is stored as 2.67499999..., so it rounds down

# Python integers are exact.
assert 2 ** 100 == 1267650600228229401496703205376

### Where you'll use it

- **Bucketing and paging:** `//` and `%` split a row number into a page and an offset, or an epoch in seconds into hour buckets.
- **Rates and averages:** `/` gives a float for a rate such as errors per request. Guard it, because a day with no traffic must not crash the job.

In [ ]:
def error_rate(errors, requests):
    if requests == 0:
        return 0.0              # a quiet day has no rate; do not divide by zero
    return errors / requests

assert error_rate(3, 4) == 0.75
assert error_rate(0, 0) == 0.0

# Epoch seconds -> hour bucket and minute within the hour.
epoch = 7_384                   # 2 hours, 3 minutes, 4 seconds
assert epoch // 3600 == 2 and (epoch % 3600) // 60 == 3

## The floating-point trap

`0.1 + 0.2` is not exactly `0.3`, because neither `0.1` nor `0.2` has an exact binary form. Never compare floats with `==` unless you know both sides are exact. Use `math.isclose` with a tolerance for measurements, and use `Decimal` for money (next section).

Rounding error also accumulates. Adding `0.1` ten times in a loop does not give `1.0`. `math.fsum` tracks the lost digits and returns the correctly rounded sum.

In [ ]:
import math

assert 0.1 + 0.2 != 0.3
assert math.isclose(0.1 + 0.2, 0.3)                  # relative tolerance, default 1e-9
assert not math.isclose(1.0, 1.001)                  # a real difference is still caught

# The stored value is the nearest binary fraction, visible with enough digits.
assert f"{0.1 + 0.2:.20f}".startswith("0.3000000000000000444")

# Accumulating in a loop loses digits; math.fsum does not.
total = 0.0
for _ in range(10):
    total += 0.1
assert total != 1.0
assert math.fsum([0.1] * 10) == 1.0

### Where you'll use it

- **Reconciliation:** comparing a total from two systems. A difference of `1e-12` is noise, so compare with a tolerance and report the largest difference.
- **Data quality rules:** a check such as "amount equals quantity times price" needs `math.isclose`, or it rejects valid rows.

## Money: use `Decimal`

`Decimal` stores decimal digits exactly, within its precision. Build it from a **string**. `Decimal(0.1)` converts the float, which already holds the binary error, so the error comes along.

Decimal arithmetic works the way a calculator does: `Decimal("0.10") + Decimal("0.20")` is exactly `Decimal("0.30")`. Round money with `quantize` and an explicit rounding mode, so the rule is written in the code.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

assert Decimal("0.10") + Decimal("0.20") == Decimal("0.30")
assert Decimal(0.1) != Decimal("0.1")                # the float error comes along
assert str(Decimal("1234.50") * 2) == "2469.00"      # the trailing zeros are kept

# quantize rounds to a fixed number of places with a named rule.
tax = Decimal("10.005")
assert tax.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP) == Decimal("10.01")

# Integer cents avoid Decimal entirely, and are common in warehouses.
price_cents = 1999
assert price_cents * 3 == 5997
assert f"{5997 // 100}.{5997 % 100:02d}" == "59.97"

### Where you'll use it

- **Finance and billing pipelines:** invoices, tax, currency conversion and ledger totals. Floats here produce totals that do not match the source system.
- **Warehouse loads:** a `DECIMAL(12,2)` column read into Python becomes a `Decimal`, which keeps its precision on the way back out.

## Strings

Strings are **immutable** sequences of Unicode characters. You cannot change one in place, so every "change" builds a new string.

- **f-strings** format values. A format spec follows a colon: `:,` adds thousands separators, `:.1%` formats a percentage, `:05d` zero-pads.
- **Methods** return new strings: `strip`, `lower`, `replace`, `split`, `join`, `startswith` and `endswith`.
- **Slices** take parts, as they do on lists. `s[::-1]` reverses a string.
- **Bytes** are a different type. `str.encode("utf-8")` turns text into bytes, and `bytes.decode("utf-8")` turns bytes back into text. Files and network data are bytes until you decode them.

In [ ]:
name = "  Ana Lopez  "
assert name.strip() == "Ana Lopez"
assert name.strip().lower() == "ana lopez"
assert name.replace(" ", "") == "AnaLopez"
assert name.count(" ") == 5          # two leading, one inside, two trailing

assert "-".join(["2024", "01", "15"]) == "2024-01-15"
assert "2024-01-15".split("-") == ["2024", "01", "15"]
assert "a,b,,c".split(",") == ["a", "b", "", "c"]      # an empty field is kept

assert f"{1234567:,}" == "1,234,567"
assert f"{0.1234:.1%}" == "12.3%"
assert f"{42:05d}" == "00042"
assert "hello"[1:4] == "ell" and "hello"[::-1] == "olleh"

# Strings cannot be changed in place.
try:
    name[0] = "X"
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

In [ ]:
# Characters and bytes differ: "é" is one character but two bytes in UTF-8.
word = "café"
assert len(word) == 4
encoded = word.encode("utf-8")
assert encoded == b"caf\xc3\xa9"
assert len(encoded) == 5
assert encoded.decode("utf-8") == word

# Decoding with the wrong encoding does not fail. It silently gives different text.
assert b"caf\xc3\xa9".decode("latin-1") == "cafÃ©"

### Where you'll use it

- **Cleaning raw text:** trim whitespace, normalise case and split delimited fields before they reach a table.
- **Building keys and paths:** `"-".join` for partition names, `f"{n:05d}"` for zero-padded file sequence numbers.
- **Encoding bugs:** text that reads as `cafÃ©` was decoded with the wrong encoding. Always name the encoding when you read or write.

## Truthiness and `None`

In a condition, these values are **falsy**: `0`, `0.0`, `""`, `[]`, `{}`, `set()` and `None`. Everything else is **truthy**. The string `"0"` and the string `"False"` are truthy, because they are not empty.

`None` means "no value", and there is only one of it. Test for it with `is None`. `if value:` treats `None`, `""` and `0` the same way, so a real zero is lost.

In [ ]:
def describe(value):
    if value is None:
        return "missing"
    if value == "":
        return "empty"
    if value == 0:
        return "zero"
    return "present"

assert describe(None) == "missing"
assert describe("") == "empty"
assert describe(0) == "zero"            # a real zero stays distinct
assert describe("0") == "present"       # the string "0" is not empty

assert not 0 and not 0.0 and not "" and not [] and not {} and not set() and not None
assert bool("False") is True            # any non-empty string is true

# None is not the string "None": str() turns it into text without an error.
assert str(None) == "None" and str(None) != None

### Where you'll use it

- **Null handling in ingestion:** a source sends `""`, `None` or a missing key. Treat each one explicitly, because a truthiness check merges them and loses a real zero.
- **Optional settings:** `if setting is None` lets a configured `0` or empty string stay valid.

## Converting between types

`int()`, `float()` and `str()` convert valid text. Invalid text raises `ValueError`. `int("12.0")` fails, because `"12.0"` is not an integer literal. Use `int(float(text))` only when you know the value is whole. The conversion step is where raw files go wrong, so it is where you should check.

In [ ]:
assert int("42") + 1 == 43
assert float("3.5") * 2 == 7.0
assert str(7) + "x" == "7x"
assert int(float("12.0")) == 12
assert int("  7 ") == 7              # int ignores surrounding whitespace
assert int("-0") == 0

for bad in ["12.0", "", "1,000"]:
    try:
        int(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"expected ValueError for {bad!r}")

### Where you'll use it

- **Parsing source files:** CSV and JSON values arrive as text. Each pipeline converts them, and the conversion is where bad rows are found.
- **API responses:** an ID sent as a string must be converted to match the integer key in the warehouse before a join works.

## Dates, times and time zones

`datetime` is a point in time, `date` is a calendar day and `timedelta` is a duration you add and subtract. Compute durations as timedeltas and convert with `.total_seconds()`; never do arithmetic on timestamp strings.

A **naive** datetime has no time zone, so it means a different instant in every place it is read. An **aware** datetime carries a `tzinfo`. Store and compare in UTC, and convert to a local zone only for display. `zoneinfo` (standard library since Python 3.9) supplies the IANA rules, including daylight-saving changes. `datetime.utcnow()` is deprecated because it returns a naive value; use `datetime.now(timezone.utc)`.

### Where you'll use it

- **Scheduling and SLAs:** "did the load finish within two hours of midnight UTC?" is a timedelta comparison between aware datetimes.
- **Event timestamps from many regions:** converting everything to UTC on ingestion makes ordering and deduplication correct.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
assert (finished - started).total_seconds() == 9000

as_new_york = finished.astimezone(ZoneInfo("America/New_York"))
assert as_new_york.hour == 10 and as_new_york.utcoffset() == timedelta(hours=-4)
assert as_new_york == finished                  # same instant, different wall clock

naive = datetime(2026, 10, 4, 12, 0)
assert naive.tzinfo is None
try:
    naive < started                             # naive and aware cannot be compared
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

### ISO 8601 parsing and the DST gap

`datetime.fromisoformat` parses ISO 8601 strings, the format APIs and warehouses return, and `.isoformat()` writes them back. Keep the offset in the string when you serialise; dropping it is how pipelines shift timestamps by hours.

Time zones with daylight saving make "add one day" ambiguous. Adding `timedelta(days=1)` to an aware local datetime keeps the wall-clock time, so across a DST change the elapsed time is 23 or 25 hours. Do duration arithmetic in UTC.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
assert parsed == datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
assert parsed.isoformat() == "2026-10-04T12:00:00+00:00"
assert parsed.date().isoformat() == "2026-10-04"

# New York leaves daylight saving on 1 November 2026.
before = datetime(2026, 10, 31, 12, 0, tzinfo=ZoneInfo("America/New_York"))
next_day = before + timedelta(days=1)
assert next_day.hour == 12                      # same wall-clock time
elapsed = next_day.astimezone(timezone.utc) - before.astimezone(timezone.utc)
assert elapsed == timedelta(hours=25)           # but 25 real hours passed

## Practice questions

1. Parse `"1,234.50"` into a `Decimal` and add `0.50`. Show the result as a string.
2. Write `mask_email(address)` that returns `"a***@example.com"` for `"ana@example.com"`. Keep the first character of the local part and the whole domain.
3. Explain in one sentence why `bool("False")` is `True`, then assert it.
4. Write `safe_ratio(part, whole)` that returns `None` when `whole` is zero, and otherwise the ratio rounded to two decimal places.

## Answers

### Question 1

Remove the thousands separator before building the `Decimal`, and pass a string, never a float.

In [ ]:
from decimal import Decimal

total = Decimal("1,234.50".replace(",", "")) + Decimal("0.50")
assert str(total) == "1235.00"

### Question 2

In [ ]:
def mask_email(address):
    local, domain = address.split("@", 1)
    return local[0] + "***@" + domain

assert mask_email("ana@example.com") == "a***@example.com"

### Question 3

`bool` on a string asks only whether it is empty. `"False"` is not empty, so it is true.

In [ ]:
assert bool("False") is True
assert bool("") is False

### Question 4

In [ ]:
def safe_ratio(part, whole):
    if whole == 0:
        return None
    return round(part / whole, 2)

assert safe_ratio(1, 3) == 0.33
assert safe_ratio(5, 0) is None

## Interview notes

- **`is` versus `==`.** `==` compares values and `is` compares identity. Use `is None` for the missing-value check, and `==` for everything else.
- **Floats are approximate.** Compare with a tolerance, and keep money in `Decimal` or integer cents.
- **`/` always gives a float**, so `total / count` is a float even when both are ints. Use `//` when you need a floor.
- **`int()` truncates, `math.floor()` floors.** They differ for negative numbers.
- **Decode bytes with a named encoding.** Most "garbled text" bugs are a wrong or missing encoding.
- **Know the type of every value you read from a file.** Most pipeline bugs start as a string where a number was expected, or as a missing value treated as empty.
- **Store UTC, display local.** Naive datetimes are ambiguous; attach `timezone.utc` and convert with `zoneinfo` only for presentation.